# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [2]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [3]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from pathlib import Path

DATA_DIR = Path("data/grade3")
assert DATA_DIR.exists(), f"Missing dataset folder: {DATA_DIR.resolve()}"

loader = DirectoryLoader(str(DATA_DIR), glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

# Keep variable name consistent with rest of notebook
synthetic_usecase_data = docs

for doc in synthetic_usecase_data:
    doc.metadata.setdefault("source", doc.metadata.get("source", "kids_science_pdf"))

Let's look at an example document to see if everything worked as expected!

In [4]:
synthetic_usecase_data[0]

Document(metadata={'producer': 'WeasyPrint 65.1', 'creator': 'ChatGPT', 'creationdate': '', 'source': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'file_path': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Grade 3 Science Booklets (Ontario Curriculum Aligned)', 'author': 'ChatGPT Deep Research', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Grade 3 Science Booklets (Ontario Curriculum\nAligned)\nBelow are outlines for five Grade 3 science booklets, each aligned with the Ontario Science & Technology\ncurriculum  (TVO  Learn).  Four  booklets  cover  the  major  science  strands  (Life  Systems,  Structures  &\nMechanisms, Matter & Energy, Earth & Space Systems), and the fifth booklet is a standalone “Just the Facts”\nreference. Each booklet is designed to be print-friendly (A4, black-and-white) and includes a

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [5]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

In [6]:
# NOTE: Disabled to preserve Tasks 4–10 learning flow.
# TODO(you): Re-enable ONLY if you remove the demo Tasks 4–10 below.
raise SystemExit("Disabled cell: retrievers are defined step-by-step in Tasks 4–10.")

# ## Cell — Define Retriever Variants (Session 9 style)
# Reuse existing variables where available: vectorstore, synthetic_usecase_data, parent_docs/text_splitter if defined

from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Base (naive) retriever from the shared Qdrant vectorstore
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
naive_retriever = base_retriever

# BM25 over the same corpus; prefer child_docs if present, otherwise original docs
_docs_for_sparse = None
try:
    _docs_for_sparse = child_docs if child_docs else None
except NameError:
    _docs_for_sparse = None
if _docs_for_sparse is None:
    _docs_for_sparse = synthetic_usecase_data
bm25_retriever = BM25Retriever.from_documents(_docs_for_sparse)

# Rerank (Contextual Compression) using Cohere Rerank
compressor = CohereRerank(model="rerank-v3.5")
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=base_retriever
)

# Multi-Query retriever using the same LLM as elsewhere (fallback if not defined yet)
try:
    _llm = chat_model
except NameError:
    _llm = ChatOpenAI(model="gpt-4.1-nano")
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever, llm=_llm
)

# Parent-Document retriever (Qdrant children + in-memory parent store)
try:
    _parent_docs = parent_docs if parent_docs else synthetic_usecase_data
except NameError:
    _parent_docs = synthetic_usecase_data

try:
    _child_splitter = text_splitter  # reuse if already defined elsewhere
except NameError:
    try:
        _child_splitter = child_splitter  # reuse if defined with a different var name
    except NameError:
        _child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

_parent_client = QdrantClient(location=":memory:")
_parent_client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)
_parent_vectorstore = QdrantVectorStore(
    collection_name="full_documents",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    client=_parent_client,
)
_parent_store = InMemoryStore()
parent_retriever = ParentDocumentRetriever(
    vectorstore=_parent_vectorstore,
    docstore=_parent_store,
    child_splitter=_child_splitter,
)
parent_retriever.add_documents(_parent_docs, ids=None)

# Ensemble retriever: combine all with equal weights
_retriever_list = [bm25_retriever, base_retriever, parent_retriever, rerank_retriever, multi_query_retriever]
_weights = [1 / len(_retriever_list)] * len(_retriever_list)
ensemble_retriever = EnsembleRetriever(retrievers=_retriever_list, weights=_weights)

print("Initialized retrievers: naive, bm25, rerank, multi_query, parent, ensemble.")


SystemExit: Disabled cell: retrievers are defined step-by-step in Tasks 4–10.

/home/lalit/workspace/code/AIE8/09_Advanced_Retrieval/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [7]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [8]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [9]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from pathlib import Path

DATA_DIR = Path("data/grade3")
assert DATA_DIR.exists(), f"Missing dataset folder: {DATA_DIR.resolve()}"

loader = DirectoryLoader(str(DATA_DIR), glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

# Keep variable name consistent with rest of notebook
synthetic_usecase_data = docs

for doc in synthetic_usecase_data:
    doc.metadata.setdefault("source", doc.metadata.get("source", "kids_science_pdf"))

Let's look at an example document to see if everything worked as expected!

In [10]:
synthetic_usecase_data[0]

Document(metadata={'producer': 'WeasyPrint 65.1', 'creator': 'ChatGPT', 'creationdate': '', 'source': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'file_path': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Grade 3 Science Booklets (Ontario Curriculum Aligned)', 'author': 'ChatGPT Deep Research', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Grade 3 Science Booklets (Ontario Curriculum\nAligned)\nBelow are outlines for five Grade 3 science booklets, each aligned with the Ontario Science & Technology\ncurriculum  (TVO  Learn).  Four  booklets  cover  the  major  science  strands  (Life  Systems,  Structures  &\nMechanisms, Matter & Energy, Earth & Space Systems), and the fifth booklet is a standalone “Just the Facts”\nreference. Each booklet is designed to be print-friendly (A4, black-and-white) and includes a

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [11]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

In [12]:
# NOTE: Disabled to preserve Tasks 4–10 learning flow.
# TODO(you): Re-enable ONLY if you remove the demo Tasks 4–10 below.
raise SystemExit("Disabled cell: retrievers are defined step-by-step in Tasks 4–10.")

# ## Cell — Define Retriever Variants (Session 9 style)
# Reuse existing variables where available: vectorstore, synthetic_usecase_data, parent_docs/text_splitter if defined

from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Base (naive) retriever from the shared Qdrant vectorstore
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
naive_retriever = base_retriever

# BM25 over the same corpus; prefer child_docs if present, otherwise original docs
_docs_for_sparse = None
try:
    _docs_for_sparse = child_docs if child_docs else None
except NameError:
    _docs_for_sparse = None
if _docs_for_sparse is None:
    _docs_for_sparse = synthetic_usecase_data
bm25_retriever = BM25Retriever.from_documents(_docs_for_sparse)

# Rerank (Contextual Compression) using Cohere Rerank
compressor = CohereRerank(model="rerank-v3.5")
rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=base_retriever
)

# Multi-Query retriever using the same LLM as elsewhere (fallback if not defined yet)
try:
    _llm = chat_model
except NameError:
    _llm = ChatOpenAI(model="gpt-4.1-nano")
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever, llm=_llm
)

# Parent-Document retriever (Qdrant children + in-memory parent store)
try:
    _parent_docs = parent_docs if parent_docs else synthetic_usecase_data
except NameError:
    _parent_docs = synthetic_usecase_data

try:
    _child_splitter = text_splitter  # reuse if already defined elsewhere
except NameError:
    try:
        _child_splitter = child_splitter  # reuse if defined with a different var name
    except NameError:
        _child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

_parent_client = QdrantClient(location=":memory:")
_parent_client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)
_parent_vectorstore = QdrantVectorStore(
    collection_name="full_documents",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
    client=_parent_client,
)
_parent_store = InMemoryStore()
parent_retriever = ParentDocumentRetriever(
    vectorstore=_parent_vectorstore,
    docstore=_parent_store,
    child_splitter=_child_splitter,
)
parent_retriever.add_documents(_parent_docs, ids=None)

# Ensemble retriever: combine all with equal weights
_retriever_list = [bm25_retriever, base_retriever, parent_retriever, rerank_retriever, multi_query_retriever]
_weights = [1 / len(_retriever_list)] * len(_retriever_list)
ensemble_retriever = EnsembleRetriever(retrievers=_retriever_list, weights=_weights)

print("Initialized retrievers: naive, bm25, rerank, multi_query, parent, ensemble.")


SystemExit: Disabled cell: retrievers are defined step-by-step in Tasks 4–10.

In [13]:
# === Eval-only helpers (do not collide with Task 8) ===
# TODO(you): eval-only; do not override Task 8 variables.
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter_eval = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)  # TODO(you): tweak if needed
parent_docs_full = synthetic_usecase_data
child_docs_eval = splitter_eval.split_documents(parent_docs_full)
print(f"[Eval helpers] parents={len(parent_docs_full)} | child_chunks={len(child_docs_eval)}")


[Eval helpers] parents=25 | child_chunks=186


## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [14]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})
base_retriever = naive_retriever  # TODO(you): shared handle for later/eval cells

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [15]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [16]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [17]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [18]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain appears to be related to Grade 3 science topics, specifically focusing on science and technology, structures, forces, soil, and environmental science. The content emphasizes concepts like building stable structures, understanding forces (push, pull, gravity, friction), soil properties, erosion, and sustainability. \n\nOverall, the primary project domain is **Grade 3 Science Education** with an emphasis on **Physical and Earth Sciences**.'

In [19]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific usecases about security mentioned. The focus appears to be on structures, their strength and stability, natural vs human-made structures, forces, and related scientific concepts suitable for Grade 3 students.'

In [20]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The context provided does not include any information about judges' opinions or comments regarding the fintech projects. Therefore, I don't know what judges had to say about the fintech projects."

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [21]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [22]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [23]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided context, the most common project domain appears to be science, specifically focusing on topics related to plants, soil, structures, forces, and Earth's environment, which are part of the Grade 3 Science curriculum. The content consistently revolves around scientific concepts such as photosynthesis, soil types, structures, forces, and basic Earth and life sciences.\n\nTherefore, the most common project domain is **Science**."

In [24]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases about security mentioned.'

In [25]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The context provided does not include any information or opinions from judges regarding fintech projects. Therefore, I do not know what judges had to say about the fintech projects.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [26]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [27]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain appears to be related to Grade 3 Science topics, particularly encompassing areas such as soil, erosion, habitats, forces, matter, light, sound, electricity, animals, and Earth sciences. The content covers various science concepts suitable for elementary education, indicating that the primary project domain is Grade 3 Science Education.'

In [29]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases mentioned about security.'

In [30]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges did not have any comments or opinions about the fintech projects in the provided context.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [31]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [32]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [33]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain based on the provided context appears to be "Structures," with a focus on understanding what makes structures strong and stable, natural versus human-made structures, and principles of stability, shapes, and materials used in construction. The content frequently references structures like buildings, bridges, nests, and the concepts of strength, stability, and supporting loads.'

In [34]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

"Based on the provided context, there are no specific use cases or examples about security discussed in the documents. The content primarily focuses on structures, forces, natural vs. human-made structures, plant biology, and environmental concepts suitable for Grade 3 students. If you need information on security use cases or related topics, I don't have that information from this context."

In [35]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The judges had positive remarks about the fintech projects, noting that the students' work demonstrated creativity, effective use of technology, and good understanding of financial concepts. They praised the students for developing innovative ideas and presenting their projects clearly. Overall, the judges appreciated the effort and ingenuity shown in the fintech projects, recognizing them as impressive examples of how young learners can engage with financial technology topics."

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [36]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [37]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [38]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [39]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [40]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [41]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain based on the provided document content appears to be science education for Grade 3 students, specifically focusing on topics related to physical science (forces and motion), structures and materials, and Earth science (soil, plants, and ecosystems). The core themes include understanding scientific concepts, exploring different materials and structures, and engaging in STEM activities related to soil and natural phenomena.'

In [42]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific usecases about security mentioned in the document. The focus is primarily on structures, their stability, materials, and their importance for safety, function, and environmental impact, but not explicitly on security.'

In [43]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The context provided does not include any information about what judges had to say regarding the fintech projects. Therefore, I do not know the answer.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [44]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [45]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [46]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, the most common project domain appears to be related to Grade 3 science topics, specifically focusing on structures, forces, soil, plants, and ecosystems. The content emphasizes understanding natural and human-made structures, their stability, materials, shapes, and importance in daily life, as well as soil and plant science.\n\nHowever, since the question asks for the "most common project domain" and the context primarily contains excerpts from science booklets covering multiple science topics, the overarching project domain is likely **Science and Technology in Grade 3** with a focus on **Structures, Forces, Soil, and Plants**.\n\nIf you are asking for a more specific, singular project domain based on the document\'s focus, it would probably be **Structures and Stability** because many excerpts discuss the importance of shapes, materials, stability, and natural versus human-made structures.\n\n**In summary:**  \n**The most common project domain is Grad

In [47]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases explicitly about security. The content focuses on forces, structures, materials, natural and human-made structures, and related scientific concepts suitable for a Grade 3 science curriculum.'

In [48]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The context provided does not include any information about what judges had to say about the fintech projects. Therefore, I don't know the answer."

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [49]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [50]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [51]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [52]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [53]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [54]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, the most common project domain appears to be "Structures" or "Building." The documents frequently discuss concepts related to structures, such as materials (wood, metal, stone), shapes (triangles, arches), stability, and importance in safety and function, especially in relation to engineering, architecture, and natural structures.'

In [55]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'There are no specific use cases about security mentioned in the provided context.'

In [56]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The context provided does not include any information about judges or their opinions regarding fintech projects. Therefore, I do not know what judges had to say about the fintech projects.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [57]:
### YOUR CODE HERE

In [58]:
# === Activity #1 Solution (Eval-only, chunk-based) ===
# TODO(you): This section is for evaluation only; students should not run demo tasks here.

from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

EMBED_MODEL = "text-embedding-3-small"
K = 10

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# Chunked eval vectorstore (separate from demo)
child_vs_eval = Qdrant.from_documents(child_docs_eval, embeddings, location=":memory:", collection_name="Kids_Science_Eval")
naive_eval = child_vs_eval.as_retriever(search_kwargs={"k": K})
bm25_eval = BM25Retriever.from_documents(child_docs_eval)

# Rerank guardrail
use_rerank = bool(os.environ.get("COHERE_API_KEY"))
if use_rerank:
    compressor = CohereRerank(model="rerank-v3.5")
    compression_eval = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=naive_eval)
else:
    compression_eval = naive_eval  # TODO(you): fallback if rerank key missing
    print("⚠️ Skipping Cohere Rerank (no COHERE_API_KEY). Using naive_eval as fallback.")

multi_query_eval = MultiQueryRetriever.from_llm(retriever=naive_eval, llm=ChatOpenAI(model="gpt-4.1-nano"))

# Parent-Doc (separate collection)
parent_client = QdrantClient(location=":memory:")
parent_client.create_collection(
    collection_name="Kids_Science_Parents_Eval",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)
parent_vs_eval = QdrantVectorStore(collection_name="Kids_Science_Parents_Eval", embedding=embeddings, client=parent_client)
parent_store = InMemoryStore()
parent_doc_eval = ParentDocumentRetriever(vectorstore=parent_vs_eval, docstore=parent_store, child_splitter=splitter_eval)
parent_doc_eval.add_documents(parent_docs_full, ids=None)

retriever_list_eval = [bm25_eval, naive_eval, compression_eval, multi_query_eval, parent_doc_eval]
ensemble_eval = EnsembleRetriever(retrievers=retriever_list_eval, weights=[1/len(retriever_list_eval)]*len(retriever_list_eval))

print("✅ Eval retrievers ready (chunked). K=10, chunk=500/50")


✅ Eval retrievers ready (chunked). K=10, chunk=500/50


In [60]:
# Sanity check the built dataset shape for one retriever (no faithfulness)
EVAL_WITH_FAITHFULNESS = False
metrics = [context_precision, context_recall]

qs = test_df["user_input"].tolist()[:3]
gt = test_df["reference"].tolist()[:3]  # string ground-truth answers
ctxs = get_contexts(naive_eval, qs, k=10)

ds = build_ragas_ds(qs, ctxs, gt)
print(ds)  # should show columns: question, contexts(list[str]), ground_truth(str)

print("Running a tiny eval…")
tiny_res = evaluate(dataset=ds, metrics=metrics)
print(tiny_res)


Dataset({
    features: ['question', 'contexts', 'ground_truth'],
    num_rows: 3
})
Running a tiny eval…


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

{'context_precision': 0.8757, 'context_recall': 1.0000}


In [65]:
# --- SDG: golden dataset ---
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_emb = LangchainEmbeddingsWrapper(embeddings)
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_emb)
# Use generate() instead of generate_with_langchain_docs() to support num_personas
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import default_transforms, apply_transforms

# Build knowledge graph
kg = KnowledgeGraph()
for doc in parent_docs_full:
    kg.nodes.append(Node(type=NodeType.DOCUMENT, properties={"page_content": doc.page_content, "document_metadata": doc.metadata}))

# Apply transforms
transforms = default_transforms(documents=parent_docs_full, llm=generator_llm, embedding_model=generator_emb)
apply_transforms(kg, transforms)

# Generate with limited personas
generator.knowledge_graph = kg
testset = generator.generate(testset_size=10, num_personas=1, raise_exceptions=False)
parent_docs_full,
testset_size=10,
num_personas=1  # Limit to 1 persona to avoid lookup bugs)
test_df = testset.to_pandas()
print("Testset preview shown below (eval-only):")
display(test_df.head())

# --- RAGAS evaluation ---
import time, pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from ragas.evaluation import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall

def format_docs(docs): return "\n\n".join(d.page_content for d in docs)
prompt = ChatPromptTemplate.from_template(
    "Use ONLY the context to answer. If unsure, say 'I don't know'.\n\nContext:\n{context}\n\nQuestion:\n{question}"
)
llm = ChatOpenAI(model="gpt-4.1-nano")
def make_chain(r):
    return (
        {"question": itemgetter("question")}
        | {"context": itemgetter("question") | r}
        | RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
        | prompt | llm | StrOutputParser()
    )

targets = {
    "Naive": naive_eval,
    "BM25": bm25_eval,
    "Rerank": compression_eval,
    "MultiQuery": multi_query_eval,
    "ParentDoc": parent_doc_eval,
    "Ensemble": ensemble_eval,
}

def get_contexts(retr, questions, k=10):
    ctxs = []
    for q in questions:
        docs = retr.get_relevant_documents(q)
        ctxs.append([d.page_content for d in docs])
    return ctxs

def build_ragas_ds(questions, contexts, ground_truth, answers=None):
    data = {"question": questions, "contexts": contexts, "ground_truth": ground_truth}
    if answers is not None:
        data["answer"] = answers
    from datasets import Dataset as HFDataset
    return HFDataset.from_dict(data)

def answer_with_chain(chain, questions):
    return [chain.invoke({"question": q}) for q in questions]

import random, numpy as np
random.seed(42); np.random.seed(42)

EVAL_WITH_FAITHFULNESS = True  # set False to skip answer generation (cheaper/faster)
metrics = [context_precision, context_recall] + ([faithfulness] if EVAL_WITH_FAITHFULNESS else [])

rows = []
for name, retr in targets.items():
    time.sleep(7)  # Cohere rate limit: 10/min
    print(f"\n🔎 Evaluating {name} ...")
    t0 = time.time()

    questions = test_df["user_input"].tolist()
    ground_truth = test_df["reference"].tolist()  # ✅ Fixed: string ground-truth, not list

    contexts = get_contexts(retr, questions, k=10)

    if EVAL_WITH_FAITHFULNESS:
        chain = make_chain(retr)
        answers = answer_with_chain(chain, questions)
        ds = build_ragas_ds(questions, contexts, ground_truth, answers)
    else:
        ds = build_ragas_ds(questions, contexts, ground_truth)

    res = evaluate(dataset=ds, metrics=metrics)

    rows.append({
        "Retriever": name,
        "ContextPrecision": float(res["context_precision"]) if isinstance(res["context_precision"], (int, float)) else float(sum(res["context_precision"]) / len(res["context_precision"])),
        "ContextRecall": float(res["context_recall"]) if isinstance(res["context_recall"], (int, float)) else float(sum(res["context_recall"]) / len(res["context_recall"])),
        "Faithfulness": float(res["faithfulness"]) if isinstance(res["faithfulness"], (int, float)) else float(sum(res["faithfulness"]) / len(res["faithfulness"])) if EVAL_WITH_FAITHFULNESS else None,
        "Latency(s)": round(time.time() - t0, 2),
    })

eval_df = pd.DataFrame(rows).sort_values(
    ["Faithfulness" if EVAL_WITH_FAITHFULNESS else "ContextRecall", "ContextPrecision"],
    ascending=False
).reset_index(drop=True)

print(f"\n📊 Activity #1 — Evaluation (Eval-only, chunked) | k=10 | chunk=500/50 | rerank={'ON' if bool(os.environ.get('COHERE_API_KEY')) else 'OFF'}")
display(eval_df)

# Sanity check on top retriever
top = eval_df.iloc[0]["Retriever"]
q = test_df.sample(1, random_state=42).iloc[0]["user_input"]
print(f"\nTop: {top}\nQ: {q}\n")
print(make_chain(targets[top]).invoke({"question": q}))


Applying HeadlinesExtractor:   0%|          | 0/24 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/25 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/47 [00:00<?, ?it/s]

Property 'summary' already exists in node 'f8673b'. Skipping!
Property 'summary' already exists in node 'e3f1e5'. Skipping!
Property 'summary' already exists in node '0e8174'. Skipping!
Property 'summary' already exists in node 'ccd142'. Skipping!
Property 'summary' already exists in node 'd95a75'. Skipping!
Property 'summary' already exists in node '937193'. Skipping!
Property 'summary' already exists in node '0d25fa'. Skipping!
Property 'summary' already exists in node 'aa20c7'. Skipping!
Property 'summary' already exists in node 'ff8875'. Skipping!
Property 'summary' already exists in node 'e3ad1d'. Skipping!
Property 'summary' already exists in node 'c4f531'. Skipping!
Property 'summary' already exists in node 'd5c24e'. Skipping!
Property 'summary' already exists in node '084c47'. Skipping!
Property 'summary' already exists in node '161b0c'. Skipping!
Property 'summary' already exists in node '5775fc'. Skipping!
Property 'summary' already exists in node '5039f2'. Skipping!
Property

...


🔎 Evaluating Ensemble ...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]


📊 Activity #1 — Evaluation (Eval-only, chunked) | k=10 | chunk=500/50 | rerank=ON


,Retriever,ContextPrecision,ContextRecall,Faithfulness,Latency(s)
0,ParentDoc,0.966667,0.800000,0.981818,31.54
1,BM25,0.516667,0.610000,0.980952,43.78
2,Naive,0.867952,0.800000,0.973333,56.87
3,Ensemble,0.746679,0.800000,0.966667,80.29
4,Rerank,1.000000,0.852857,0.962500,48.30
5,MultiQuery,0.821813,0.800000,0.895238,61.81



Top: ParentDoc
Q: Whay is loam soil considered good for plant growthing and how can you tell if soil is loam?

Loam soil is considered good for plant growth because it holds nutrients and water effectively while also draining well and providing air for roots. You can tell if soil is loam by its appearance and feel: it is usually dark, soft, crumbly, and easy to dig. A simple test is the feel test—moist loam soil will form a short ribbon that breaks apart easily when rolled into a ball and then into a "snake," indicating a balanced mix of sand, silt, and clay with organic matter.


In [66]:
# === Eval-only helpers (do not collide with Task 8) ===
# TODO(you): eval-only; do not override Task 8 variables.
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter_eval = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)  # TODO(you): tweak if needed
parent_docs_full = synthetic_usecase_data
child_docs_eval = splitter_eval.split_documents(parent_docs_full)
print(f"[Eval helpers] parents={len(parent_docs_full)} | child_chunks={len(child_docs_eval)}")


[Eval helpers] parents=25 | child_chunks=186


## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [67]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})
base_retriever = naive_retriever  # TODO(you): shared handle for later/eval cells

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [68]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [69]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [70]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [71]:
naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided content, the most common project domain appears to be science education related to the Grade 3 Ontario curriculum. Specifically, the focus is on topics such as structures, forces and motion, soil and earth science, plant growth, and environmental sustainability. The documents include activities, diagrams, key concepts like stability, materials, forces, and ecosystems, all aimed at engaging students in science learning at an elementary level.'

In [72]:
naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided content, there are no specific use cases about security mentioned.'

In [73]:
naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The provided context does not include any specific comments or opinions from judges regarding the fintech projects. Therefore, I don't know what judges had to say about the fintech projects."

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [74]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [75]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [76]:
bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

"Based on the provided context, the most common project domain appears to be related to Grade 3 Science topics, specifically focusing on Earth's natural features and processes, plant biology, structures, and basic physical science concepts. The documents include science booklets covering topics like plants and sunlight, soil and environment, structures, and fundamental physical sciences such as force, matter, and light. These topics are aligned with the Ontario Grade 3 Science curriculum, indicating that the primary project domain is educational science content for third-grade students."

In [77]:
bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases or discussions related to security.'

In [78]:
bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The context does not provide any information about what judges had to say about the fintech projects.'

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [79]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [80]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [81]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, which includes multiple references to soil, nutrients, erosion, structures, forces, and related scientific concepts, it appears that the most common project domain is related to science, specifically earth science and physical science topics such as soil, erosion, plant needs, forces, and structures.'

In [82]:
contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific use cases about security mentioned.'

In [83]:
contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The judges did not have any comments or opinions regarding the fintech projects, as the provided documents focus on science topics such as forces, structures, soil, and ecosystems for grade 3 students.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [84]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [85]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [86]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'Based on the provided context, the most common project domain appears to be related to "structures," specifically focusing on how to make them strong and stable. The documents mainly discuss natural and human-made structures, principles of stability, materials, shapes like triangles, and their importance in everyday life, safety, and environmental considerations. \n\nTherefore, the most common project domain is:\n\n**Structures and Structural Stability**\n\nIf you need a more specific focus within that domain, it centers on understanding what makes structures strong and stable, including materials, shapes, and natural vs. built structures.'

In [87]:
multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are references to security in the general discussion of structures, such as buildings, bridges, and supports needing to be strong and stable to prevent collapse and accidents. The importance of designing structures that can handle forces and loads to ensure safety is highlighted. However, there are no specific use cases explicitly about security measures, cybersecurity, or security technologies.\n\nTherefore, the answer is:  \n**There are no specific use cases about security mentioned in the provided text.**'

In [88]:
multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The context provided does not include any specific commentary or opinions from judges regarding the fintech projects. Therefore, I do not know what judges had to say about the fintech projects.'

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [89]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [90]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [91]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [92]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [93]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [94]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain based on the provided context appears to be science education for Grade 3 students, with specific focus areas including physical science (forces, motion, structures, materials), Earth science (soil types, ecosystems), and biological science (animal life cycles, habitats). The content revolves around curriculum-aligned science topics, activities, and key concepts suitable for Grade 3 learners.'

In [95]:
parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there are no specific usecases about security mentioned. The focus is primarily on the importance of strong and stable structures for safety, function, and environmental considerations.'

In [96]:
parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

"The provided context does not include any information about what judges had to say about the fintech projects. Therefore, I don't know the judges' opinions or comments regarding those projects."

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [97]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [98]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [99]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain appears to be related to Grade 3 science topics, specifically focused on science and technology curriculum aligned with the Ontario Curriculum. The prevalent themes include structures (natural and human-made), forces and motion, soil and environment, plants and ecosystems, and healthy living. Many of the referenced materials, activities, and resources emphasize understanding structures, forces, materials, soil conservation, and plant biology, indicating that the primary project domain centers on elementary science education, especially physical and earth sciences at the Grade 3 level.'

In [100]:
ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided content, there are no explicit use cases or examples about security. The document primarily discusses topics related to forces, structures, materials, natural and human-made structures, and environmental considerations, without specifically addressing security applications.'

In [101]:
ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'Judges had to say that structures are everywhere and their strength and stability are crucial for safety and function. They emphasized that natural and human-made structures both need to be strong and stable to serve their purpose. For example, structures like towers, dams, and buildings are designed with principles such as wide bases, low centers of gravity, and the use of strong materials and shapes (like triangles) to resist forces like wind and weight. The judges also highlighted the importance of understanding how forces like gravity, wind, and tension affect structures, and how natural structures (like nests or beaver dams) inspire human engineering. Overall, the judges emphasized that strong, stable structures are vital for safety, functionality, and environmental considerations.'

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [102]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [103]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [104]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [105]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [106]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [107]:
semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

'The most common project domain appears to be related to "Structures," specifically focusing on understanding and designing different types of structures, their materials, stability, shapes, and their importance in safety, function, and environmental impact.'

In [108]:
semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

'Based on the provided context, there do not appear to be any specific use cases or discussions about security.'

In [109]:
semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

'The context does not provide any information about what judges had to say regarding the fintech projects. Therefore, I do not know the answer.'

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [ ]:
### YOUR CODE HERE

In [110]:
# === Activity #1 Solution (Eval-only, chunk-based) ===
# TODO(you): This section is for evaluation only; students should not run demo tasks here.

from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

EMBED_MODEL = "text-embedding-3-small"
K = 10

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# Chunked eval vectorstore (separate from demo)
child_vs_eval = Qdrant.from_documents(child_docs_eval, embeddings, location=":memory:", collection_name="Kids_Science_Eval")
naive_eval = child_vs_eval.as_retriever(search_kwargs={"k": K})
bm25_eval = BM25Retriever.from_documents(child_docs_eval)

# Rerank guardrail
use_rerank = bool(os.environ.get("COHERE_API_KEY"))
if use_rerank:
    compressor = CohereRerank(model="rerank-v3.5")
    compression_eval = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=naive_eval)
else:
    compression_eval = naive_eval  # TODO(you): fallback if rerank key missing
    print("⚠️ Skipping Cohere Rerank (no COHERE_API_KEY). Using naive_eval as fallback.")

multi_query_eval = MultiQueryRetriever.from_llm(retriever=naive_eval, llm=ChatOpenAI(model="gpt-4.1-nano"))

# Parent-Doc (separate collection)
parent_client = QdrantClient(location=":memory:")
parent_client.create_collection(
    collection_name="Kids_Science_Parents_Eval",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)
parent_vs_eval = QdrantVectorStore(collection_name="Kids_Science_Parents_Eval", embedding=embeddings, client=parent_client)
parent_store = InMemoryStore()
parent_doc_eval = ParentDocumentRetriever(vectorstore=parent_vs_eval, docstore=parent_store, child_splitter=splitter_eval)
parent_doc_eval.add_documents(parent_docs_full, ids=None)

retriever_list_eval = [bm25_eval, naive_eval, compression_eval, multi_query_eval, parent_doc_eval]
ensemble_eval = EnsembleRetriever(retrievers=retriever_list_eval, weights=[1/len(retriever_list_eval)]*len(retriever_list_eval))

print("✅ Eval retrievers ready (chunked). K=10, chunk=500/50")


✅ Eval retrievers ready (chunked). K=10, chunk=500/50


In [112]:
# === Test Cell: Check model access ===
print("Testing model access...\n")

# Test 1: Direct ChatOpenAI (no wrapper)
print("1️⃣ Testing gpt-4.1-nano directly (no RAGAS wrapper)...")
try:
    from langchain_openai import ChatOpenAI
    direct_llm = ChatOpenAI(model="gpt-4.1-nano")
    response = direct_llm.invoke("Say hi")
    print(f"✅ SUCCESS: {response.content}\n")
except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}: {str(e)[:100]}\n")

# Test 2: With RAGAS wrapper
print("2️⃣ Testing gpt-4.1-nano with RAGAS LangchainLLMWrapper...")
try:
    from ragas.llms import LangchainLLMWrapper
    wrapped_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
    response = wrapped_llm.generate_text("Say hi")
    print(f"✅ SUCCESS: {response}\n")
except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}: {str(e)[:100]}\n")

# Test 3: Fallback model (gpt-4o-mini)
print("3️⃣ Testing gpt-4o-mini with RAGAS wrapper (fallback)...")
try:
    fallback_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
    response = fallback_llm.generate_text("Say hi")
    print(f"✅ SUCCESS: {response}\n")
except Exception as e:
    print(f"❌ FAILED: {type(e).__name__}: {str(e)[:100]}\n")

print("=" * 60)
print("RECOMMENDATION:")
print("- If Test 1 ✅ but Test 2 ❌ → Use gpt-4o-mini (wrapper issue)")
print("- If Test 1 ❌ → API key/billing issue")
print("- If Test 3 ✅ → Safe to use gpt-4o-mini as replacement")

Testing model access...

1️⃣ Testing gpt-4.1-nano directly (no RAGAS wrapper)...
✅ SUCCESS: Hi!

2️⃣ Testing gpt-4.1-nano with RAGAS LangchainLLMWrapper...
❌ FAILED: AttributeError: 'str' object has no attribute 'to_messages'

3️⃣ Testing gpt-4o-mini with RAGAS wrapper (fallback)...
❌ FAILED: AttributeError: 'str' object has no attribute 'to_messages'

RECOMMENDATION:
- If Test 1 ✅ but Test 2 ❌ → Use gpt-4o-mini (wrapper issue)
- If Test 1 ❌ → API key/billing issue
- If Test 3 ✅ → Safe to use gpt-4o-mini as replacement


In [114]:
# --- SDG: golden dataset ---
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_emb = LangchainEmbeddingsWrapper(embeddings)
testset = TestsetGenerator(llm=generator_llm, embedding_model=generator_emb)\
    .generate_with_langchain_docs(parent_docs_full, testset_size=10)
test_df = testset.to_pandas()
print("Testset preview shown below (eval-only):")
display(test_df.head())

# --- RAGAS evaluation ---
import time, pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from ragas.evaluation import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall

def format_docs(docs): return "\n\n".join(d.page_content for d in docs)
prompt = ChatPromptTemplate.from_template(
    "Use ONLY the context to answer. If unsure, say 'I don't know'.\n\nContext:\n{context}\n\nQuestion:\n{question}"
)
llm = ChatOpenAI(model="gpt-4.1-nano")
def make_chain(r):
    return (
        {"question": itemgetter("question")}
        | {"context": itemgetter("question") | r}
        | RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
        | prompt | llm | StrOutputParser()
    )

targets = {
    "Naive": naive_eval,
    "BM25": bm25_eval,
    "Rerank": compression_eval,
    "MultiQuery": multi_query_eval,
    "ParentDoc": parent_doc_eval,
    "Ensemble": ensemble_eval,
}

def get_contexts(retr, questions, k=10):
    ctxs = []
    for q in questions:
        docs = retr.get_relevant_documents(q)
        ctxs.append([d.page_content for d in docs])
    return ctxs

def build_ragas_ds(questions, contexts, ground_truth, answers=None):
    data = {"question": questions, "contexts": contexts, "ground_truth": ground_truth}
    if answers is not None:
        data["answer"] = answers
    from datasets import Dataset as HFDataset
    return HFDataset.from_dict(data)

def answer_with_chain(chain, questions):
    return [chain.invoke({"question": q}) for q in questions]

import random, numpy as np
random.seed(42); np.random.seed(42)

EVAL_WITH_FAITHFULNESS = True  # set False to skip answer generation (cheaper/faster)
metrics = [context_precision, context_recall] + ([faithfulness] if EVAL_WITH_FAITHFULNESS else [])

rows = []
for name, retr in targets.items():
    time.sleep(7)  # Cohere rate limit: 10/min
    print(f"\n🔎 Evaluating {name} ...")
    t0 = time.time()

    questions = test_df["user_input"].tolist()
    ground_truth = test_df["reference"].tolist()  # ✅ Fixed: string ground-truth, not list

    contexts = get_contexts(retr, questions, k=10)

    if EVAL_WITH_FAITHFULNESS:
        chain = make_chain(retr)
        answers = answer_with_chain(chain, questions)
        ds = build_ragas_ds(questions, contexts, ground_truth, answers)
    else:
        ds = build_ragas_ds(questions, contexts, ground_truth)

    res = evaluate(dataset=ds, metrics=metrics)

    rows.append({
        "Retriever": name,
        "ContextPrecision": float(res["context_precision"]) if isinstance(res["context_precision"], (int, float)) else float(sum(res["context_precision"]) / len(res["context_precision"])),
        "ContextRecall": float(res["context_recall"]) if isinstance(res["context_recall"], (int, float)) else float(sum(res["context_recall"]) / len(res["context_recall"])),
        "Faithfulness": float(res["faithfulness"]) if isinstance(res["faithfulness"], (int, float)) else float(sum(res["faithfulness"]) / len(res["faithfulness"])) if EVAL_WITH_FAITHFULNESS else None,
        "Latency(s)": round(time.time() - t0, 2),
    })

eval_df = pd.DataFrame(rows).sort_values(
    ["Faithfulness" if EVAL_WITH_FAITHFULNESS else "ContextRecall", "ContextPrecision"],
    ascending=False
).reset_index(drop=True)

print(f"\n📊 Activity #1 — Evaluation (Eval-only, chunked) | k=10 | chunk=500/50 | rerank={'ON' if bool(os.environ.get('COHERE_API_KEY')) else 'OFF'}")
display(eval_df)

# Sanity check on top retriever
top = eval_df.iloc[0]["Retriever"]
q = test_df.sample(1, random_state=42).iloc[0]["user_input"]
print(f"\nTop: {top}\nQ: {q}\n")
print(make_chain(targets[top]).invoke({"question": q}))


Applying HeadlinesExtractor:   0%|          | 0/24 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/25 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/47 [00:00<?, ?it/s]

Property 'summary' already exists in node '1991bb'. Skipping!
Property 'summary' already exists in node '3d7f8c'. Skipping!
Property 'summary' already exists in node '00c155'. Skipping!
Property 'summary' already exists in node '42f583'. Skipping!
Property 'summary' already exists in node '83b34c'. Skipping!
Property 'summary' already exists in node '1ee75a'. Skipping!
Property 'summary' already exists in node '4c1eb1'. Skipping!
Property 'summary' already exists in node '23d6f9'. Skipping!
Property 'summary' already exists in node '788cdd'. Skipping!
Property 'summary' already exists in node '75c211'. Skipping!
Property 'summary' already exists in node '3c816b'. Skipping!
Property 'summary' already exists in node 'fd39ec'. Skipping!
Property 'summary' already exists in node 'cb8668'. Skipping!
Property 'summary' already exists in node '10ffa1'. Skipping!
Property 'summary' already exists in node '5fbed9'. Skipping!
Property 'summary' already exists in node 'bbf2a7'. Skipping!
Property

...


🔎 Evaluating Ensemble ...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]


📊 Activity #1 — Evaluation (Eval-only, chunked) | k=10 | chunk=500/50 | rerank=ON


,Retriever,ContextPrecision,ContextRecall,Faithfulness,Latency(s)
0,ParentDoc,1.000000,1.00,0.918333,32.91
1,MultiQuery,0.849744,1.00,0.915556,57.52
2,Ensemble,0.742216,1.00,0.864762,82.57
3,Rerank,1.000000,0.95,0.833333,29.91
4,Naive,0.910095,1.00,0.740873,29.70
5,BM25,0.400000,0.60,0.309524,19.52



Top: ParentDoc
Q: How does rice cultivation adapt to clay soil conditions, and why is rice often grown in flooded fields within clay soils?

Rice cultivation adapts to clay soil conditions because rice plants can tolerate flooded fields, which are often created in clay soils. Rice is grown in flooded fields called rice paddies because the water prevents roots from suffocating or rotting in the heavy clay, which otherwise has poor aeration and drainage. The flooded water helps the rice thrive despite the challenging soil conditions.
